# **1. INSTALL LIBRARY**

In [95]:
# Streamlit → UI chatbot / web app
!pip install -q streamlit

# Gemini SDK → communication with the Gemini API (AI model)
!pip install -q google-genai

# Embedding + similarity → semantic search / similar context retrieval
!pip install -q sentence-transformers scikit-learn

# Vector database → storing embeddings for memory / RAG
!pip install -q chromadb

# Create a public URL from Colab → so Streamlit can be accessed in the browser
!pip install -q pyngrok

## **2. IMPORT LIBRARY & BASIC SETUP**

In [96]:
# =====================================================================================
# IMPORT LIBRARY
# =====================================================================================

# Operating system utilities
import os

# JSON data handling
import json

# Unique ID generator
import uuid

# Streamlit framework for web app UI
import streamlit as st

# Gemini API
from google import genai

# Embedding Model
from sentence_transformers import SentenceTransformer

# Similarity calculation
from sklearn.metrics.pairwise import cosine_similarity

# Vector Database
import chromadb



# =====================================================================================
# LOAD EMBEDDING MODEL
# =====================================================================================

# Cache the model so it loads only once
@st.cache_resource

# Function to load embedding model
def load_embedding_model():

    # Load sentence transformer model
    return SentenceTransformer(
        'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    )

# Initialize embedding model
embedding_model = load_embedding_model()

# Display success message
print("✅ Model embedding telah dimuat.")



# =====================================================================================
# CREATE VECTOR DATABASE
# =====================================================================================

# Create persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# RESET OLD COLLECTION
try:

    # Delete old collection if it exists
    chroma_client.delete_collection(
        "campus_helpdesk_memory"
    )

    # Display success message
    print("✅ Collection lama berhasil dihapus.")

except:
    # Display message if collection does not exist
    print("❌ Collection belum ada.")

# Create or load collection for chatbot memory
collection = chroma_client.get_or_create_collection(
    name="campus_helpdesk_memory",
    metadata={"hnsw:space": "cosine"}
)

# Display success message
print("✅ ChromaDB telah diinisialisasi.")



# =====================================================================================
# USER MEMORY
# =====================================================================================

# Dictionary for storing user session memory
user_memory = {}

# Display success message
print("✅ Memori pengguna telah diinisialisasi.")



# =====================================================================================
# TOXIC WORD LIST
# =====================================================================================

# List of toxic or inappropriate words
TOXIC_WORDS = [
    "goblok", "tolol", "anjing", "bodoh", "jembut", "bajingan",
    "bangsat", "kampret", "tai", "kontol", "memek", "idiot",
    "jancok", "asu", "setan", "brengsek", "keparat", "bacot",
    "ngentot", "monyet", "sialan", "laknat", "tai kucing",
    "pecundang", "sampah", "bebal", "otak udang", "tolol banget",
    "gila", "edan", "sinting", "kampungan", "cacat", "bejat"
]

# Display success message
print("✅ Filter kata Toxic telah diinisialisasi.")



# =====================================================================================
# SYSTEM STATUS
# =====================================================================================

# Display chatbot startup status
print("\n🚀 Campus Helpdesk (Chatbot) telah disiapkan.")

✅ Model embedding telah dimuat.
✅ Collection lama berhasil dihapus.
✅ ChromaDB telah diinisialisasi.
✅ Memori pengguna telah diinisialisasi.
✅ Filter kata Toxic telah diinisialisasi.

🚀 Campus Helpdesk (Chatbot) telah disiapkan.


# **3. SETUP GEMINI API KEY**

In [109]:
# Access secret data stored in Google Colab
from google.colab import userdata


# =====================================================================================
# LOAD SECRET FROM COLAB
# =====================================================================================

# Store Gemini API key into environment variable
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI")



# =====================================================================================
# LOAD API KEY FROM ENVIRONMENT VARIABLE
# =====================================================================================

# Retrieve Gemini API key from environment variable
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")



# =====================================================================================
# GEMINI CLIENT
# =====================================================================================

# Create Gemini API client
client = genai.Client(api_key=GEMINI_API_KEY)

# Display secure connection success message
print("✅ Gemini API berhasil dikoneksikan secara aman.")

✅ Gemini API berhasil dikoneksikan secara aman.


# **4. CAMPUS DATASET (KNOWLEDGE BASE)**

In [108]:
# Dataset containing campus knowledge base information
campus_data = [


    # =====================================================================================
    # UNIVERSITY
    # =====================================================================================

    # University information
    {
        "category": "universitas",
        "title": "Universitas Teknologi Nusantara",
        "keywords": [
            "universitas",
            "kampus terbaik",
            "kampus teknologi"
        ],
        "content": """
        Universitas Teknologi Nusantara merupakan
        universitas berbasis teknologi dan AI.

        Universitas memiliki:
        - Fakultas Teknik
        - Fakultas Bisnis
        - Fakultas Desain

        Akreditasi Universitas: Unggul
        Lokasi: Surabaya
        """
    },



    # =====================================================================================
    # STUDY PROGRAM
    # =====================================================================================

    # Informatics study program information
    {
        "category": "program_studi",
        "title": "Teknik Informatika",
        "keywords": [
            "informatika",
            "AI",
            "programming",
            "coding"
        ],
        "content": """
        Program Studi Teknik Informatika fokus pada:
        - Artificial Intelligence
        - Machine Learning
        - Data Science
        - Web Development
        - Mobile Development
        - Cyber Security

        Akreditasi: A
        Jenjang: S1

        Prospek karier:
        - Software Engineer
        - AI Engineer
        - Data Scientist
        - Backend Developer
        """
    },

    {
        "category": "program_studi",
        "title": "Sistem Informasi",
        "keywords": [
            "sistem informasi",
            "business",
            "data analytics"
        ],
        "content": """
        Program Studi Sistem Informasi fokus pada:
        - Business Intelligence
        - ERP
        - UI/UX
        - Data Analytics
        - Project Management

        Akreditasi: A
        Jenjang: S1

        Prospek karier:
        - System Analyst
        - IT Consultant
        - Data Analyst
        """
    },



    # =====================================================================================
    # FACULTY
    # =====================================================================================

    # Faculty information
    {
        "category": "fakultas",
        "title": "Fakultas Teknik",
        "keywords": [
            "fakultas",
            "teknik"
        ],
        "content": """
        Fakultas Teknik menaungi:
        - Teknik Informatika
        - Sistem Informasi
        - Teknik Industri

        Fakultas memiliki:
        - Laboratorium AI
        - Laboratorium Komputer
        - Ruang Multimedia
        """
    },



    # =====================================================================================
    # UKT / TUITION FEE
    # =====================================================================================

    # Tuition fee information
    {
        "category": "ukt",
        "title": "UKT Teknik Informatika",
        "keywords": [
            "ukt",
            "biaya",
            "informatika"
        ],
        "content": """
        UKT Teknik Informatika berkisar:
        - Rp 5.000.000
        - Rp 7.500.000

        per semester tergantung jalur masuk.
        """
    },

    {
        "category": "ukt",
        "title": "UKT Sistem Informasi",
        "keywords": [
            "ukt",
            "biaya",
            "sistem informasi"
        ],
        "content": """
        UKT Sistem Informasi berkisar:
        - Rp 4.500.000
        - Rp 6.500.000

        per semester tergantung jalur masuk.
        """
    },



    # =====================================================================================
    # SCHOLARSHIP
    # =====================================================================================

    # Scholarship information
    {
        "category": "beasiswa",
        "title": "Beasiswa Prestasi Akademik",
        "keywords": [
            "beasiswa",
            "prestasi",
            "ipk"
        ],
        "content": """
        Beasiswa akademik diberikan kepada mahasiswa
        dengan IPK minimal 3.5.

        Benefit:
        - Potongan UKT 50%
        - Bantuan penelitian
        - Sertifikat prestasi
        """
    },

    {
        "category": "beasiswa",
        "title": "Beasiswa Organisasi",
        "keywords": [
            "beasiswa organisasi",
            "organisasi"
        ],
        "content": """
        Beasiswa organisasi diberikan kepada mahasiswa
        aktif organisasi kampus.

        Benefit:
        - Potongan UKT
        - Dana kegiatan mahasiswa
        """
    },



    # =====================================================================================
    # FACILITIES
    # =====================================================================================

    # Campus facilities information
    {
        "category": "fasilitas",
        "title": "Fasilitas Kampus",
        "keywords": [
            "fasilitas",
            "laboratorium",
            "wifi"
        ],
        "content": """
        Kampus memiliki fasilitas:
        - WiFi kampus
        - Laboratorium komputer
        - Perpustakaan digital
        - Studio multimedia
        - Coworking space
        - Auditorium
        - Kantin mahasiswa
        """
    },



    # =====================================================================================
    # ACCREDITATION
    # =====================================================================================

    # Accreditation information
    {
        "category": "akreditasi",
        "title": "Akreditasi Kampus",
        "keywords": [
            "akreditasi",
            "unggul"
        ],
        "content": """
        Universitas Teknologi Nusantara memiliki:
        - Akreditasi Universitas: Unggul

        Program Studi:
        - Teknik Informatika: A
        - Sistem Informasi: A
        """
    },



    # =====================================================================================
    # CAREER
    # =====================================================================================

    # Career opportunities information
    {
        "category": "karier",
        "title": "Prospek Karier Lulusan",
        "keywords": [
            "karier",
            "pekerjaan",
            "lulusan"
        ],
        "content": """
        Lulusan kampus memiliki prospek karier:
        - Software Engineer
        - AI Engineer
        - Data Scientist
        - UI/UX Designer
        - System Analyst
        - IT Consultant
        """
    },



    # =====================================================================================
    # REGISTRATION
    # =====================================================================================

    # Student registration information
    {
        "category": "pendaftaran",
        "title": "Pendaftaran Mahasiswa Baru",
        "keywords": [
            "pendaftaran",
            "daftar kuliah",
            "mahasiswa baru"
        ],
        "content": """
        Persyaratan pendaftaran:
        - Fotokopi ijazah
        - KTP
        - Pas foto
        - Mengisi formulir online

        Jalur pendaftaran:
        - Reguler
        - Prestasi
        - Beasiswa
        """
    },



    # =====================================================================================
    # CURRICULUM
    # =====================================================================================

    # Curriculum information
    {
        "category": "kurikulum",
        "title": "Kurikulum Teknik Informatika",
        "keywords": [
            "kurikulum",
            "mata kuliah",
            "informatika"
        ],
        "content": """
        Mata kuliah unggulan:
        - Artificial Intelligence
        - Machine Learning
        - Deep Learning
        - Basis Data
        - Pemrograman Web
        - Mobile Development
        """
    },



    # =====================================================================================
    # CAMPUS LOCATION
    # =====================================================================================

    # Campus location information
    {
        "category": "lokasi",
        "title": "Lokasi Kampus",
        "keywords": [
            "lokasi",
            "alamat",
            "surabaya"
        ],
        "content": """
        Kampus berlokasi di:
        Surabaya, Jawa Timur.

        Akses lokasi:
        - Dekat terminal
        - Dekat stasiun
        - Area pusat pendidikan
        """
    }

]

# Display dataset creation success message
print("✅ Dataset kampus berhasil dibuat.")



# =====================================================================================
# ENTER INTO THE VECTOR DATABASE
# =====================================================================================

# Check if database is still empty
if collection.count() == 0:

    # Loop through all campus data
    for data in campus_data:

        # Combine all important text into one string
        full_text = f"""
        Kategori: {data['category']}
        Judul: {data['title']}
        Keywords: {', '.join(data['keywords'])}

        {data['content']}
        """

        # Convert text into embedding vector
        embedding = embedding_model.encode(
            full_text
        ).tolist()

        # Store data into ChromaDB collection
        collection.add(
            ids=[str(uuid.uuid4())],
            embeddings=[embedding],
            documents=[full_text],
            metadatas=[{
                "category": data["category"],
                "title": data["title"],
                "keywords": ", ".join(data["keywords"])
            }]
        )

    # Display success message after inserting data
    print("✅ Data masuk ke ChromaDB.")

# If database already contains data
else:

    # Display database already initialized message
    print("✅ ChromaDB sudah memiliki data.")



# =====================================================================================
# TOTAL DATA
# =====================================================================================

# Display total knowledge base data
print("\nTOTAL DATA KNOWLEDGE BASE:")
print(len(campus_data))

✅ Dataset kampus berhasil dibuat.
✅ ChromaDB sudah memiliki data.

TOTAL DATA KNOWLEDGE BASE:
14


# **5. TOXIC FILTER SYSTEM**

In [99]:
# Maximum allowed toxic score before user is blocked
MAX_TOXIC_SCORE = 3


# =====================================================================================
# INITIALIZE USER MEMORY
# =====================================================================================

# Create user memory if user does not exist
def initialize_user(user_id):

    # Check whether user already exists
    if user_id not in user_memory:

        # Initialize default user data
        user_memory[user_id] = {
            "history": [],
            "topics": [],
            "repeated_questions": [],
            "toxic_score": 0,
            "warnings": []
        }



# =====================================================================================
# DETECT TOXIC MESSAGE
# =====================================================================================

# Detect toxic words inside user message
def detect_toxicity(text):

    # Convert text into lowercase
    text = text.lower()

    # Store detected toxic words
    detected_words = []

    # Loop through toxic word list
    for word in TOXIC_WORDS:

        # Check whether toxic word exists in text
        if word in text:

            # Save detected toxic word
            detected_words.append(word)

    # Return all detected toxic words
    return detected_words



# =====================================================================================
# HANDLE TOXIC USER
# =====================================================================================

# Handle toxic behavior from user
def handle_toxicity(user_id, toxic_words):

    # Increase toxic score
    user_memory[user_id]["toxic_score"] += 1

    # Get current toxic score
    toxic_score = user_memory[user_id]["toxic_score"]


    # SAVE WARNING HISTORY
    # Save moderation warning history
    user_memory[user_id]["warnings"].append({
        "toxic_words": toxic_words,
        "level": toxic_score
    })


    # LEVEL 1
    # First warning level
    if toxic_score == 1:

        # Return warning response
        return """
⚠️ Mohon gunakan bahasa yang sopan saat menggunakan layanan Campus Helpdesk.

Saya tetap siap membantu Anda dalam menemukan informasi terkait:
- Universitas
- Jurusan
- UKT
- Beasiswa
- Kurikulum
- Fasilitas kampus
"""


    # LEVEL 2
    # Second warning level
    elif toxic_score == 2:

        # Return stronger warning response
        return """
⚠️ Terdeteksi penggunaan bahasa yang tidak sesuai.

Agar percakapan tetap nyaman dan profesional,
mohon gunakan bahasa yang lebih baik.

Silakan lanjutkan pertanyaan terkait kebutuhan akademik atau informasi kampus.
"""


    # LEVEL 3 (BLOCK USER)
    # Block user after exceeding toxic limit
    elif toxic_score >= MAX_TOXIC_SCORE:

        # Return blocked message
        return """
⛔ Percakapan sementara dihentikan karena terdeteksi penggunaan bahasa yang melanggar etika layanan kampus.

Silakan coba kembali nanti dengan bahasa yang lebih sopan dan konstruktif.
"""



# =====================================================================================
# CHECK USER BLOCKED
# =====================================================================================

# Check whether user is blocked
def is_user_blocked(user_id):

    # Return block status based on toxic score
    return (
        user_memory[user_id]["toxic_score"]
        >= MAX_TOXIC_SCORE
    )



# =====================================================================================
# RESET TOXIC SCORE
# =====================================================================================

# Reset user moderation status
def reset_toxic_score(user_id):

    # Reset toxic score back to zero
    user_memory[user_id]["toxic_score"] = 0

    # Clear warning history
    user_memory[user_id]["warnings"] = []



# =====================================================================================
# SHOW USER MODERATION STATUS
# =====================================================================================

# Display user moderation information
def show_moderation_status(user_id):

    # Print moderation title
    print("\n🛡️ STATUS MODERASI")

    # Display toxic score
    print(
        f"Nilai Toxic: "
        f"{user_memory[user_id]['toxic_score']}"
    )

    # Display total warnings
    print(
        f"Total Peringatan: "
        f"{len(user_memory[user_id]['warnings'])}"
    )



# =====================================================================================
# TEST TOXIC FILTER
# =====================================================================================

# Sample test user
test_user = "demo_user"

# Initialize test user memory
initialize_user(test_user)

# Example toxic message
sample_text = "kampus ini goblok"

# Detect toxic words from sample text
detected_words = detect_toxicity(sample_text)

# Check whether toxic words are found
if detected_words:

    # Generate moderation response
    response = handle_toxicity(
        test_user,
        detected_words
    )

    # Display toxic detection message
    print("🚨 TERDETEKSI PESAN TOXIC\n")

    # Display detected toxic words
    print("Kata-kata yang Terdeteksi:")
    print(detected_words)

    # Display moderation response
    print("\nRespon:")
    print(response)

# If no toxic words detected
else:

    # Display safe message status
    print("✅ Pesan Aman.")



# =====================================================================================
# SHOW STATUS
# =====================================================================================

# Display final moderation status
show_moderation_status(test_user)

🚨 TERDETEKSI PESAN TOXIC

Kata-kata yang Terdeteksi:
['goblok']

Respon:

⚠️ Mohon gunakan bahasa yang sopan saat menggunakan layanan Campus Helpdesk.

Saya tetap siap membantu Anda dalam menemukan informasi terkait:
- Universitas
- Jurusan
- UKT
- Beasiswa
- Kurikulum
- Fasilitas kampus


🛡️ STATUS MODERASI
Nilai Toxic: 1
Total Peringatan: 1


# **6. MEMORY & REPEATED QUESTION SYSTEM**

In [100]:
# Similarity threshold for repeated question detection
SIMILARITY_THRESHOLD = 0.80

# Maximum number of chat history records
MAX_HISTORY = 20


# =====================================================================================
# SAVE CHAT HISTORY
# =====================================================================================

# Save user question into memory
def save_to_memory(user_id, question):

    # LIMIT CHAT HISTORY
    # Check if history exceeds maximum limit
    if len(user_memory[user_id]["history"]) >= MAX_HISTORY:

        # Remove oldest chat history
        user_memory[user_id]["history"].pop(0)


    # SAVE QUESTION
    # Store new question into history
    user_memory[user_id]["history"].append(question)



# =====================================================================================
# GET QUESTION EMBEDDING
# =====================================================================================

# Convert text into embedding vector
def get_embedding(text):

    # Generate embedding using sentence transformer
    return embedding_model.encode(text)



# =====================================================================================
# DETECT REPEATED QUESTION
# =====================================================================================

# Detect whether question is repeated
def is_repeated_question(user_id, new_question):

    # Get user chat history
    history = user_memory[user_id]["history"]


    # EMPTY HISTORY
    # Return false if history is empty
    if len(history) == 0:

        # No repeated question detected
        return False, None


    # NEW QUESTION EMBEDDING
    # Generate embedding for new question
    new_embedding = get_embedding(
        new_question
    )

    # Store highest similarity score
    best_similarity = 0

    # Store most similar question
    most_similar_question = None


    # CHECK ALL HISTORY
    # Compare new question with all previous questions
    for old_question in history:

        # Generate embedding for old question
        old_embedding = get_embedding(
            old_question
        )

        # Calculate cosine similarity
        similarity = cosine_similarity(
            [new_embedding],
            [old_embedding]
        )[0][0]


        # SAVE BEST MATCH
        # Save highest similarity result
        if similarity > best_similarity:

            # Update best similarity score
            best_similarity = similarity

            # Save most similar question
            most_similar_question = old_question


    # SIMILAR QUESTION DETECTED
    # Check if similarity exceeds threshold
    if best_similarity >= SIMILARITY_THRESHOLD:

        # Save repeated question information
        user_memory[user_id][
            "repeated_questions"
        ].append({
            "question": new_question,
            "similar_to": most_similar_question,
            "similarity_score": float(best_similarity)
        })

        # Return repeated question result
        return True, best_similarity

    # Return non-repeated result
    return False, best_similarity



# =====================================================================================
# REPEATED QUESTION RESPONSE
# =====================================================================================

# Generate response for repeated question
def repeated_question_response(similarity_score):

    # Convert similarity score into percentage
    similarity_percent = round(
        similarity_score * 100,
        1
    )

    # Return repeated question message
    return f"""
🔁 Pertanyaan serupa terdeteksi.

Kemiripan pertanyaan:
{similarity_percent}%

Silakan lihat jawaban sebelumnya terlebih dahulu agar informasi tidak berulang.

Jika ingin informasi lebih detail atau berbeda,
silakan ajukan pertanyaan yang lebih spesifik.
"""



# =====================================================================================
# TRACK USER INTEREST
# =====================================================================================

# Track topics that interest the user
def track_user_interest(user_id, text):

    # Convert text into lowercase
    text = text.lower()

    # List of important keywords
    keywords = [

        # Study program related keywords
        "informatika",
        "sistem informasi",
        "data science",
        "cyber security",
        "ai",

        # Campus related keywords
        "universitas",
        "fakultas",
        "kampus",

        # Academic related keywords
        "kurikulum",
        "mata kuliah",
        "akreditasi",

        # Administration related keywords
        "biaya",
        "ukt",
        "beasiswa",
        "pendaftaran",

        # Facility related keywords
        "laboratorium",
        "wifi",
        "perpustakaan",

        # Career related keywords
        "karier",
        "prospek kerja",
        "pekerjaan"
    ]


    # SAVE USER INTEREST
    # Detect and save user interests
    for keyword in keywords:

        # Check whether keyword exists in text
        if keyword in text:

            # Avoid duplicate topics
            if keyword not in user_memory[user_id]["topics"]:

                # Save keyword into user topics
                user_memory[user_id]["topics"].append(
                    keyword
                )



# =====================================================================================
# GET USER FAVORITE TOPICS
# =====================================================================================

# Get all user interest topics
def get_user_topics(user_id):

    # Return user topics list
    return user_memory[user_id]["topics"]



# =====================================================================================
# SHOW USER PROFILE
# =====================================================================================

# Display user profile information
def show_user_profile(user_id):

    # Get user profile data
    profile = user_memory[user_id]

    # Print profile title
    print("\n👤 PROFIL PENGGUNA")

    # Display profile statistics
    print(f"""
Jumlah Histori                  : {len(profile['history'])}
Topik yang Menarik              : {profile['topics']}
Pertanyaan yang Sering Diajukan : {len(profile['repeated_questions'])}
Nilai Toxic                     : {profile['toxic_score']}
""")



# =====================================================================================
# CLEAR USER MEMORY
# =====================================================================================

# Reset all user memory data
def clear_user_memory(user_id):

    # Reset user memory into default state
    user_memory[user_id] = {
        "history": [],
        "topics": [],
        "repeated_questions": [],
        "toxic_score": 0,
        "warnings": []
    }



# =====================================================================================
# TEST MEMORY SYSTEM
# =====================================================================================

# Sample user for testing
test_user = "mahasiswa_001"

# Initialize sample user
initialize_user(test_user)

# First sample question
question_1 = (
    "berapa biaya kuliah teknik informatika?"
)

# Second sample question
question_2 = (
    "biaya teknik informatika berapa ya?"
)

# Third sample question
question_3 = (
    "apakah ada beasiswa kampus?"
)



# =====================================================================================
# SAVE FIRST QUESTION
# =====================================================================================

# Save first question into memory
save_to_memory(
    test_user,
    question_1
)



# =====================================================================================
# CHECK REPEATED QUESTION
# =====================================================================================

# Detect repeated question similarity
is_repeat, similarity_score = (
    is_repeated_question(
        test_user,
        question_2
    )
)

# Display repeated question checking title
print("\n🔍 PEMERIKSAAN PERTANYAAN YANG BERULANG")

# Display repeated detection result
print("Diulangi:", is_repeat)

# Check if repeated question detected
if is_repeat:

    # Display repeated question response
    print(
        repeated_question_response(
            similarity_score
        )
    )



# =====================================================================================
# TRACK USER INTEREST
# =====================================================================================

# Track user interest from first question
track_user_interest(
    test_user,
    question_1
)

# Track user interest from third question
track_user_interest(
    test_user,
    question_3
)



# =====================================================================================
# SHOW PROFILE
# =====================================================================================

# Display user profile summary
show_user_profile(test_user)


🔍 PEMERIKSAAN PERTANYAAN YANG BERULANG
Diulangi: True

🔁 Pertanyaan serupa terdeteksi.

Kemiripan pertanyaan:
84.80000305175781%

Silakan lihat jawaban sebelumnya terlebih dahulu agar informasi tidak berulang.

Jika ingin informasi lebih detail atau berbeda,
silakan ajukan pertanyaan yang lebih spesifik.


👤 PROFIL PENGGUNA

Jumlah Histori                  : 1
Topik yang Menarik              : ['informatika', 'biaya', 'kampus', 'beasiswa']
Pertanyaan yang Sering Diajukan : 1
Nilai Toxic                     : 0



# **7. SEMANTIC SEARCH + RAG SYSTEM**

In [113]:
# Top number of relevant documents to retrieve
TOP_K_RESULTS = 3

# Minimum similarity score required to consider a result relevant
MIN_RELEVANCE_SCORE = 0.68


# =====================================================================================
# SEARCH RELEVANT KNOWLEDGE
# =====================================================================================

# Search the vector database for relevant information based on user question
def search_knowledge(user_question):


    # QUESTION EMBEDDING
    # Convert user question into embedding vector
    question_embedding = embedding_model.encode(
        user_question
    ).tolist()


    # SEARCH VECTOR DATABASE
    # Query ChromaDB using embedding similarity search
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=TOP_K_RESULTS
    )

    # Extract retrieved documents
    documents = results["documents"][0]

    # Extract metadata of retrieved documents
    metadatas = results["metadatas"][0]

    # Extract similarity distances
    distances = results["distances"][0]

    # Store filtered knowledge results
    knowledge = []


    # PROCESS SEARCH RESULTS
    # Iterate through retrieved results
    for doc, meta, distance in zip(
        documents,
        metadatas,
        distances
    ):

        # CONVERT DISTANCE TO SCORE
        # Convert distance into relevance score (higher is better)
        relevance_score = 1 / (1 + distance)


        # SKIP LOW RELEVANCE
        # Ignore results below threshold
        if relevance_score < MIN_RELEVANCE_SCORE:

            # Skip irrelevant result
            continue

        # Store relevant knowledge item
        knowledge.append({

            "title": meta["title"],
            "category": meta["category"],
            "keywords": meta.get(
                "keywords",
                ""
            ),
            "content": doc,
            "relevance_score": round(
                relevance_score,
                3
            )
        })

    # Return filtered and ranked knowledge results
    return knowledge



# =====================================================================================
# BUILD RAG CONTEXT
# =====================================================================================

# Build formatted context string from retrieved knowledge
def build_context(knowledge):


    # EMPTY KNOWLEDGE
    # Handle case when no relevant data is found
    if len(knowledge) == 0:

        # Return fallback message
        return """
Tidak ditemukan informasi yang relevan
di knowledge base kampus.
"""

    # Initialize context string
    context = ""


    # BUILD CONTEXT
    # Format each knowledge item into readable context
    for index, item in enumerate(
        knowledge,
        start=1
    ):

        # Append formatted knowledge block
        context += f"""
==============================
DATA {index}
==============================

Kategori:
{item['category']}

Judul:
{item['title']}

Keywords:
{item['keywords']}

Relevance Score:
{item['relevance_score']}

Isi Informasi:
{item['content']}

"""

    # Return final context string
    return context



# =====================================================================================
# CREATE FINAL RAG PROMPT
# =====================================================================================

# Build final prompt for LLM using retrieved knowledge
def build_rag_prompt(user_question):


    # SEARCH KNOWLEDGE
    # Retrieve relevant documents from vector database
    knowledge = search_knowledge(
        user_question
    )


    # BUILD CONTEXT
    # Convert retrieved knowledge into structured context
    context = build_context(
        knowledge
    )


    # FINAL PROMPT
    # Construct prompt for AI model response generation
    final_prompt = f"""
Kamu adalah AI Campus Helpdesk
yang membantu mahasiswa mencari
informasi akademik dan kampus.

Jawablah berdasarkan knowledge base berikut.

========================================
KNOWLEDGE BASE
========================================

{context}

========================================
ATURAN MENJAWAB
========================================

- Jawab dengan sopan dan profesional
- Gunakan bahasa Indonesia
- Fokus pada informasi kampus
- Jangan membuat informasi palsu
- Jika informasi tidak tersedia,
  katakan dengan jujur
- Buat jawaban jelas dan terstruktur

========================================
PERTANYAAN USER
========================================

{user_question}

========================================
JAWABAN AI
========================================
"""

    # Return final constructed prompt
    return final_prompt



# =====================================================================================
# SHOW SEARCH RESULT
# =====================================================================================

# Display retrieved search results in console
def show_search_result(knowledge):

    # Print header
    print("\n📚 HASIL PENCARIAN\n")

    # Handle empty results
    if len(knowledge) == 0:

        # Print no results message
        print("Tidak ada data relevan")

        # Exit function
        return

    # Iterate through results
    for index, item in enumerate(
        knowledge,
        start=1
    ):

        # Print formatted result
        print(
            f"""
================================
HASIL {index}
================================

Kategori :
{item['category']}

Judul :
{item['title']}

Relevance :
{item['relevance_score']}

Isi :
{item['content']}
"""
        )



# =====================================================================================
# TEST RAG SYSTEM
# =====================================================================================

# Sample user question for testing
question = (
    "berapa biaya kuliah teknik informatika?"
)



# =====================================================================================
# SEARCH KNOWLEDGE
# =====================================================================================

# Retrieve relevant knowledge for test question
knowledge_result = search_knowledge(
    question
)



# =====================================================================================
# SHOW RESULT
# =====================================================================================

# Display retrieved knowledge results
show_search_result(
    knowledge_result
)



# =====================================================================================
# BUILD FINAL PROMPT
# =====================================================================================

# Generate final RAG prompt
final_prompt = build_rag_prompt(
    question
)



# =====================================================================================
# PREVIEW PROMPT
# =====================================================================================

# Print preview of generated prompt
print("\n🧠 FINAL PROMPT RAG\n")

# Display first part of prompt for inspection
print(final_prompt[:2500])


📚 HASIL PENCARIAN


HASIL 1

Kategori :
ukt

Judul :
UKT Teknik Informatika

Relevance :
0.756

Isi :

        Kategori: ukt
        Judul: UKT Teknik Informatika
        Keywords: ukt, biaya, informatika

        
        UKT Teknik Informatika berkisar:
        - Rp 5.000.000
        - Rp 7.500.000

        per semester tergantung jalur masuk.
        
        


HASIL 2

Kategori :
ukt

Judul :
UKT Sistem Informasi

Relevance :
0.7

Isi :

        Kategori: ukt
        Judul: UKT Sistem Informasi
        Keywords: ukt, biaya, sistem informasi

        
        UKT Sistem Informasi berkisar:
        - Rp 4.500.000
        - Rp 6.500.000

        per semester tergantung jalur masuk.
        
        


🧠 FINAL PROMPT RAG


Kamu adalah AI Campus Helpdesk
yang membantu mahasiswa mencari
informasi akademik dan kampus.

Jawablah berdasarkan knowledge base berikut.

KNOWLEDGE BASE


DATA 1

Kategori:
ukt

Judul:
UKT Teknik Informatika

Keywords:
ukt, biaya, informatika

Relevance Score:
0

# **8. SYSTEM PROMPT + AI RESPONSE ENGINE**

In [114]:
# System prompt that defines AI behavior, role, and response rules
SYSTEM_PROMPT = """
Kamu adalah AI Campus Helpdesk modern di Indonesia.

Tugas utama kamu:
- Membantu mahasiswa dan calon mahasiswa
- Menjawab pertanyaan akademik
- Menjawab administrasi kampus
- Memberikan informasi universitas
- Memberikan informasi jurusan
- Memberikan informasi UKT
- Memberikan informasi beasiswa
- Memberikan informasi fasilitas kampus
- Memberikan informasi akreditasi
- Memberikan informasi karier lulusan

Aturan penting:
- Gunakan bahasa Indonesia
- Gunakan bahasa formal dan profesional
- Bersikap seperti staf akademik universitas
- Jangan menggunakan bahasa kasar
- Jangan membuat informasi palsu
- Jika informasi tidak tersedia,
  katakan dengan jujur
- Fokus pada bantuan akademik dan kampus
- Jika pertanyaan di luar topik kampus,
  tolak dengan sopan

Gaya jawaban:
- Jelas
- Ringkas
- Informatif
- Mudah dipahami
- Terstruktur
"""



# =====================================================================================
# OUT OF TOPIC DETECTION
# =====================================================================================

# Detect whether user input is outside campus-related topics
def is_out_of_topic(text):

    # Convert text to lowercase for matching
    text = text.lower()

    # List of campus-related keywords
    campus_keywords = [

        # University-related keywords
        "kampus",
        "universitas",
        "fakultas",
        "jurusan",
        "program studi",

        # Academic-related keywords
        "kuliah",
        "mahasiswa",
        "dosen",
        "kurikulum",
        "mata kuliah",

        # Administrative-related keywords
        "ukt",
        "biaya",
        "beasiswa",
        "pendaftaran",
        "akreditasi",

        # Campus facility-related keywords
        "laboratorium",
        "perpustakaan",
        "wifi",

        # Career-related keywords
        "karier",
        "prospek kerja",
        "lulusan",

        # Study program-related keywords
        "informatika",
        "sistem informasi",
        "data science",
        "cyber security",
        "ai"
    ]

    # Check if any campus keyword exists in input text
    for keyword in campus_keywords:

        # If keyword found, consider it on-topic
        if keyword in text:

            # Not out of topic
            return False

    # If no keyword matches, it is out of topic
    return True



# =====================================================================================
# OUT OF TOPIC RESPONSE
# =====================================================================================

# Response when user asks unrelated questions
def out_of_topic_response():

    # Return predefined refusal message
    return """
Maaf, saya hanya dapat membantu terkait informasi kampus dan akademik.

Silakan tanyakan hal seperti:
- Universitas
- Jurusan
- UKT
- Kurikulum
- Beasiswa
- Fasilitas kampus
- Akreditasi
- Prospek karier
"""



# =====================================================================================
# GENERATE RECOMMENDATION
# =====================================================================================

# Generate personalized recommendations based on user interests
def generate_recommendation(user_id):

    # Retrieve user interest topics
    topics = user_memory[user_id]["topics"]

    # Initialize recommendation list
    recommendations = []


    # STUDY PROGRAM
    if "informatika" in topics:
        recommendations.append(
            "📚 Program Studi Teknik Informatika"
        )

    if "sistem informasi" in topics:
        recommendations.append(
            "📚 Program Studi Sistem Informasi"
        )


    # ACADEMIC
    if "kurikulum" in topics:
        recommendations.append(
            "🧠 Informasi Kurikulum dan Mata Kuliah"
        )

    if "akreditasi" in topics:
        recommendations.append(
            "🌐 Informasi Akreditasi Kampus"
        )


    # ADMINISTRATION
    if "biaya" in topics or "ukt" in topics:
        recommendations.append(
            "💰 Informasi UKT dan Pembayaran"
        )

    if "beasiswa" in topics:
        recommendations.append(
            "📖 Informasi Beasiswa Kampus"
        )

    if "pendaftaran" in topics:
        recommendations.append(
            "📝 Informasi Pendaftaran Mahasiswa Baru"
        )


    # FACILITY
    if "laboratorium" in topics:
        recommendations.append(
            "🏢 Informasi Laboratorium Kampus"
        )


    # CAREER
    if "karier" in topics:
        recommendations.append(
            "🚀 Prospek Karier Lulusan"
        )


    # EMPTY RECOMMENDATION
    # Return empty string if no recommendations
    if len(recommendations) == 0:
        return ""

    # Build recommendation text header
    recommendation_text = """

========================================
REKOMENDASI UNTUK ANDA
========================================
"""

    # Format recommendations list
    for index, item in enumerate(
        recommendations,
        start=1
    ):

        # Append each recommendation
        recommendation_text += (
            f"\n{index}. {item}"
        )

    # Return final recommendation string
    return recommendation_text



# =====================================================================================
# GENERATE AI RESPONSE
# =====================================================================================

# Main function to generate AI response using RAG + Gemini
def generate_ai_response(
    user_id,
    user_question
):

    # INITIALIZE USER
    # Ensure user memory exists
    initialize_user(user_id)


    # USER BLOCKED
    # Check if user is blocked due to toxicity
    if is_user_blocked(user_id):

        # Return blocked message
        return """
⛔ Akses percakapan dibatasi sementara
karena terdeteksi pelanggaran etika komunikasi.
"""


    # TOXIC DETECTION
    # Detect toxic words in user input
    toxic_words = detect_toxicity(
        user_question
    )

    # Handle toxic input if detected
    if toxic_words:

        # Return moderation response
        return handle_toxicity(
            user_id,
            toxic_words
        )


    # OUT OF TOPIC DETECTION
    # Check if question is outside campus scope
    if is_out_of_topic(user_question):

        # Return out-of-topic response
        return out_of_topic_response()


    # REPEATED QUESTION DETECTION
    # Check if user repeats similar question
    is_repeat, similarity_score = (
        is_repeated_question(
            user_id,
            user_question
        )
    )

    # Handle repeated question
    if is_repeat:

        # Return repeated question response
        return repeated_question_response(
            similarity_score
        )


    # TRACK USER INTEREST
    # Store user topic preferences
    track_user_interest(
        user_id,
        user_question
    )


    # SEARCH KNOWLEDGE
    # Retrieve relevant context from vector database
    knowledge = search_knowledge(
        user_question
    )


    # BUILD CONTEXT
    # Convert retrieved knowledge into readable format
    context = build_context(
        knowledge
    )


    # BUILD FINAL RAG PROMPT
    # Construct prompt for Gemini model
    final_prompt = f"""
{SYSTEM_PROMPT}

========================================
KNOWLEDGE BASE
========================================

{context}

========================================
PERTANYAAN USER
========================================

{user_question}

========================================
INSTRUKSI MENJAWAB
========================================

- Jawab berdasarkan knowledge base
- Jangan membuat informasi palsu
- Gunakan bahasa Indonesia
- Gunakan format rapi
- Jika ada angka atau biaya,
  tampilkan dengan jelas
- Jika informasi tidak tersedia,
  katakan dengan jujur

========================================
JAWABAN AI
========================================
"""


    # GENERATE GEMINI RESPONSE
    # Call Gemini API to generate answer
    try:

        # Request response from Gemini model
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=final_prompt
        )

        # Extract AI response text
        ai_answer = response.text


    # HANDLE API ERROR
    # Catch API-related exceptions
    except Exception as e:

        # Convert exception to string
        error_message = str(e)


        # API LIMIT
        if (
            "429" in error_message
            or
            "RESOURCE_EXHAUSTED"
            in error_message
        ):

            # Return rate limit message
            ai_answer = """
Permintaan sementara tidak dapat diproses
karena limit penggunaan API.

Silakan coba beberapa saat lagi.
"""


        # API KEY ERROR
        elif "API key" in error_message:

            # Return invalid API key message
            ai_answer = """
API Key Gemini tidak valid.
"""


        # GENERAL ERROR
        else:

            # Return generic error message
            ai_answer = """
Terjadi kesalahan saat memproses permintaan.
Silakan coba kembali.
"""


    # SAVE USER INTERACTION
    # Store question into memory
    save_to_memory(
        user_id,
        user_question
    )


    # ADD PERSONALIZED RECOMMENDATION
    # Append recommendation based on user interests
    ai_answer += generate_recommendation(
        user_id
    )

    # Return final response
    return ai_answer



# =====================================================================================
# TEST AI RESPONSE ENGINE
# =====================================================================================

# Create test user ID
test_user = "mahasiswa_demo"

# Sample question for testing
question = (
    "berapa biaya kuliah teknik informatika?"
)

# Generate AI response
answer = generate_ai_response(
    test_user,
    question
)

# Display AI response output
print("\n🎓 RESPON AI\n")

# Print final answer
print(answer)


🎓 RESPON AI


🔁 Pertanyaan serupa terdeteksi.

Kemiripan pertanyaan:
100.0%

Silakan lihat jawaban sebelumnya terlebih dahulu agar informasi tidak berulang.

Jika ingin informasi lebih detail atau berbeda,
silakan ajukan pertanyaan yang lebih spesifik.



# **9. THEME SETTINGS**

In [103]:
# Import operating system utilities
import os

# Import Path for file system handling
from pathlib import Path


# =====================================================================================
# SELECT THEME
# =====================================================================================

# Set UI theme mode (default is "light")
# Comment the line below to switch to dark mode
THEME_MODE = "light"

# Uncomment the line below to switch to dark mode
# THEME_MODE = "dark"



# =====================================================================================
# AUTO CREATE .streamlit/config.toml
# =====================================================================================

# Create .streamlit directory if it does not exist
os.makedirs(".streamlit", exist_ok=True)

# Streamlit configuration file content
config = f'''
[theme]
base="{THEME_MODE}"

[client]
toolbarMode = "minimal"
showSidebarNavigation = false
'''

# Write configuration into config.toml file
with open(".streamlit/config.toml", "w", encoding="utf-8") as f:

    # Save configuration to file
    f.write(config)



# =====================================================================================
# AUTO UPDATE css_loader.py
# =====================================================================================

# Determine which CSS file should be active based on theme
if THEME_MODE == "light":
    active_css = "light_mode.css"

# If theme is not light, use dark mode CSS
else:
    active_css = "dark_mode.css"

# Generate Python file content for CSS loader
css_loader_content = f'''from pathlib import Path

ACTIVE_THEME = "{active_css}"

def load_all_css():
    # Define the folder that contains all CSS files
    css_dir = Path(__file__).parent / "styles"

    # List of CSS files that will be loaded in order
    css_files = [
        "base.css",
        ACTIVE_THEME
    ]

    # Variable to store all combined CSS content
    all_css = ""

    # Loop through each CSS file
    for file in css_files:
        # Create full file path
        path = css_dir / file

        # Check if the file exists before reading it
        if path.exists():
            # Read file content and append it to all_css
            all_css += path.read_text(encoding="utf-8") + "\\n"

    # Return all combined CSS content
    return all_css
'''

# File path for CSS loader script
loader_path = "css_loader.py"

# Write updated CSS loader file
with open(loader_path, "w", encoding="utf-8") as f:
    f.write(css_loader_content)

# Display active theme information
print("Tema yang sedang aktif:", THEME_MODE)

Tema yang sedang aktif: light


# **10. STREAMLIT RUNTIME CONFIGURATION**



In [104]:
# Import ngrok tunneling service for public access
from pyngrok import ngrok

# Import secret storage from Google Colab
from google.colab import userdata

# Import subprocess for running external processes
import subprocess

# Import time module for delays and sleep control
import time



# =====================================================================================
# NGROK AUTH
# =====================================================================================

# Retrieve ngrok authentication token from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')

# Authenticate ngrok with the provided token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Store reference to running Streamlit process
proc = None

# Store reference to active ngrok tunnel
tunnel = None



# =====================================================================================
# START STREAMLIT
# =====================================================================================

# Function to start Streamlit app and expose it via ngrok
def run_streamlit(filename, port=8501):

    # Access global process and tunnel variables
    global proc, tunnel

    # Stop any previously running Streamlit instance
    stop_streamlit()

    # Start Streamlit application using subprocess
    proc = subprocess.Popen(
        [
            "streamlit",
            "run",
            filename,
            "--server.port",
            str(port),
            "--server.headless",
            "true"
        ]
    )

    # Wait for Streamlit server to initialize
    time.sleep(5)

    # Create new ngrok tunnel for public access
    tunnel = ngrok.connect(port)

    # Display Streamlit public URL
    print("\n🚀 URL Streamlit:")

    # Print generated ngrok URL
    print(tunnel.public_url)

    # Return public URL
    return tunnel.public_url



# =====================================================================================
# STOP STREAMLIT
# =====================================================================================

# Function to stop Streamlit and close ngrok tunnel
def stop_streamlit():

    # Access global process and tunnel variables
    global proc, tunnel


    # Safely close ngrok tunnel if it exists
    try:

        # Check if tunnel is active
        if tunnel is not None:

            # Disconnect ngrok tunnel
            ngrok.disconnect(tunnel.public_url)

            # Reset tunnel reference
            tunnel = None

             # Confirmation message
            print("✅ Tunnel Ngrok diputuskan.")

    # Handle tunnel disconnection errors
    except Exception as e:

        # Print error message
        print("Ngrok disconnect:", e)


    # Attempt to kill ngrok process
    try:

        # Stop all ngrok processes
        ngrok.kill()

    # Handle ngrok kill errors
    except Exception as e:

        # Print error message
        print("Ngrok kill:", e)


    # Stop Streamlit subprocess if running
    try:

        # Check if process exists
        if proc is not None:

            # Terminate Streamlit process
            proc.terminate()

            # Wait for process to fully stop
            proc.wait(timeout=5)

            # Reset process reference
            proc = None

            # Confirmation message
            print("✅ Streamlit telah dihentikan.")

    # Handle Streamlit termination errors
    except Exception as e:

        # Print error message
        print("Streamlit stop:", e)

    # Force cleanup of any remaining Streamlit processes
    subprocess.run(
        ["pkill", "-f", "streamlit"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # Wait to ensure cleanup is completed
    time.sleep(3)

    # Final cleanup confirmation message
    print("✅ Proses pembersihan telah selesai.")

# **11. START APP MANUALLY**

In [105]:
proc = run_streamlit("app.py")

✅ Proses pembersihan telah selesai.

🚀 URL Streamlit:
https://dazzler-dyslexia-repackage.ngrok-free.dev


# **12. STOP APP MANUALLY**

In [106]:
# Open the code below to manually stop the application

# stop_streamlit()  # Stop the Streamlit server
# time.sleep(3)     # Wait for cleanup process to finish